# --- Plan A: annotation-based relevance ---

In [13]:


# EDIT THESE:
CONFIRMED_CSV   = "verification_summary_1 (test).csv"  # neuron_id, ecii concepts
ANNOTATIONS_DIR = "test_annotations"                     # root folder with class subdirs of SUN2012 XMLs
REAL_ACT_CSV    = "preds_of_64Neurons_denseLayer_test.csv"                    # 64 cols "0".."63" + filenames
FAKE_ACT_CSV    = "[img+obj_labels_to_fakes]fake_activations(test).csv"                    # same schema; filenames may be 'fake_<...>.jpg'

# optional small synonym map you can extend
SYNONYMS = {
    "sofa": {"couch"},
    "dish rack": {"dish_rack"},
    "toilet tissue": {"toilet_tissue"},
}


In [14]:
DEBUG = True
N_SHOW = 5  # how many examples to print in each preview


In [15]:
import os, re, csv, xml.etree.ElementTree as ET
import numpy as np

def canon_phrase(s: str) -> str:
    """Canonical whole-phrase form: lowercase, underscores->spaces, collapse spaces."""
    if s is None: return ""
    s = s.strip().lower()
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s)
    return s

def split_multi_concepts(raw: str) -> list[str]:
    """Split multiple concepts ONLY on '_and_'; do NOT split into tokens."""
    if raw is None: return []
    parts = [p for p in raw.split("_and_") if p.strip()]
    return parts if parts else [raw]

# Normalize SYNONYMS map to canonical phrases
SYNONYMS = {canon_phrase(k): {canon_phrase(v) for v in vs} for k, vs in SYNONYMS.items()}

def phrases_from_label(raw: str, synonyms_map=None) -> set[str]:
    """
    From a raw confirmed label, return a set of WHOLE phrases.
    - Split only on '_and_'
    - Canonicalize each phrase
    - Expand via whole-phrase synonyms (if provided)
    """
    out = set()
    for piece in split_multi_concepts(raw):
        p = canon_phrase(piece)
        if not p:
            continue
        out.add(p)
        if synonyms_map and p in synonyms_map:
            out |= {canon_phrase(x) for x in synonyms_map[p]}
    return out


def load_confirmed_neuron_labels(confirmed_csv_path,
                                 id_col_candidates=("neuron_id","neuron","id"),
                                 label_col_candidates=("ecii_concepts","label","name"),
                                 synonyms_map=None):
    """
    Returns: dict {neuron_id:int -> set(WHOLE-PHRASE strings)}
    """
    out = {}
    with open(confirmed_csv_path, "r", newline="", encoding="utf-8") as f:
        rdr = csv.DictReader(f)
        header = rdr.fieldnames or []
        id_col  = next((c for c in id_col_candidates  if c in header), None)
        lab_col = next((c for c in label_col_candidates if c in header), None)
        if id_col is None or lab_col is None:
            raise RuntimeError(f"Missing ID/label columns in {confirmed_csv_path}")

        for row in rdr:
            try:
                j = int(float(row[id_col]))
            except:
                continue
            if not (0 <= j <= 63):
                continue
            phs = phrases_from_label(row[lab_col], synonyms_map=synonyms_map)
            if phs:
                out[j] = phs

    if not out:
        raise RuntimeError("No confirmed neurons parsed from CSV.")
    return out

def parse_sun_xml_objects(xml_path, synonyms_map=None) -> set[str]:
    """
    Parse a SUN2012-style XML and return a set of WHOLE-PHRASE object names.
    """
    try:
        root = ET.parse(xml_path).getroot()
    except Exception:
        return set()

    phrases = set()
    for obj in root.findall("./object"):
        nm = obj.findtext("name") or ""
        for piece in split_multi_concepts(nm):
            p = canon_phrase(piece)
            if not p:
                continue
            phrases.add(p)
            if synonyms_map and p in synonyms_map:
                phrases |= {canon_phrase(x) for x in synonyms_map[p]}
    return phrases


In [16]:
def preview_confirmed_neurons(confirmed_dict, n=N_SHOW):
    """
    confirmed_dict: {neuron_id: set(label tokens)}
    """
    print(f"[preview] confirmed neurons: total={len(confirmed_dict)}")
    for k in sorted(confirmed_dict)[:n]:
        print(f"  neuron {k}: labels={sorted(list(confirmed_dict[k]))[:8]}")

def compute_threshold_from_real_rows(real_rows, neuron_cols, ratio=0.8):
    first = True
    max_act = None
    for r in real_rows:
        vals = np.array([float(r[c]) for c in neuron_cols], dtype=np.float64)
        if first:
            max_act = vals; first = False
        else:
            max_act = np.maximum(max_act, vals)
    return ratio * max_act

def preview_thresholds(threshold, confirmed_dict=None, n=N_SHOW):
    """
    threshold: np.array shape (64,)
    shows a few example T_j; if confirmed_dict given, show those IDs first.
    """
    print(f"[preview] per-neuron thresholds (len={threshold.size})")
    ids = list(range(threshold.size))
    if confirmed_dict:
        ids = sorted(confirmed_dict.keys()) + [i for i in ids if i not in confirmed_dict]
    for j in ids[:n]:
        lab = sorted(list(confirmed_dict.get(j, [])))[:3] if confirmed_dict else []
        print(f"  j={j:02d}  T_j={threshold[j]:.4f}  labels={lab}")

def preview_fake_matching(real_rows, fake_idx, real_fname_col, n=N_SHOW):
    print("[preview] real→fake filename mapping attempts")
    shown = 0
    for rr in real_rows:
        fname = rr[real_fname_col]
        tried = _candidate_fake_keys(fname)
        hit = next((k for k in tried if k in fake_idx), None)
        print(f"  real='{fname}'")
        for t in tried:
            mark = "✓" if t == hit else "·"
            print(f"    {mark} try: {t}")
        shown += 1
        if shown >= n: break

def preview_image_objects_and_matches(annotations_root, confirmed, image_relpaths, n=N_SHOW):
    print("[preview] XML objects & confirmed-label matches (first few images)")
    for fn in image_relpaths[:n]:
        objs = load_image_objects_map(annotations_root, [fn]).get(fn, set())
        print(f"  image: {fn}")
        print(f"    objects: {sorted(list(objs))[:12]}")
        hits = []
        misses = []
        for j, labset in confirmed.items():
            if labset & objs:
                hits.append(j)
            else:
                misses.append(j)
        print(f"    relevant confirmed (by annotation): {sorted(hits)[:12]}  (count={len(hits)})")
        print(f"    irrelevant confirmed: {sorted(misses)[:12]}  (count={len(misses)})")


In [17]:
def image_to_xml_path(annotations_root, img_relpath):
    """
    images: 'classname/filename.jpg'
    xml:    'annotations_root/classname/filename.xml'
    """
    d = os.path.dirname(img_relpath)
    b = os.path.splitext(os.path.basename(img_relpath))[0] + ".xml"
    return os.path.join(annotations_root, d, b)

def load_image_objects_map(annotations_root, image_relpaths, synonyms_map=None):
    """Return {filename -> set(WHOLE-PHRASE objects)}."""
    m = {}
    for fn in image_relpaths:
        xmlp = image_to_xml_path(annotations_root, fn)
        m[fn] = parse_sun_xml_objects(xmlp, synonyms_map=synonyms_map)
    return m


In [18]:
def load_csv_as_dicts(path):
    with open(path, "r", newline="", encoding="utf-8") as f:
        rdr = csv.DictReader(f)
        rows = [r for r in rdr]
        header = rdr.fieldnames
    return header, rows

def find_filename_column(header):
    for cand in ("filenames","filename","file","path"):
        if cand in header:
            return cand
    raise RuntimeError("No filename column found (filenames/filename/file/path).")

def ensure_neuron_columns(header, n=64):
    cols = [str(i) for i in range(n)]
    miss = [c for c in cols if c not in header]
    if miss:
        raise RuntimeError(f"Missing neuron columns, e.g., {miss[:5]}")
    return cols

def _candidate_fake_keys(orig_fname):
    """
    Build possible keys for the fake CSV when fakes are named like 'fake_<basename>.jpg'.
    Tries (in order):
      1) dir/fake_basename
      2) fake_<relpath>
      3) fake_basename
      4) original relpath
    """
    d = os.path.dirname(orig_fname)
    b = os.path.basename(orig_fname)
    pref_b = f"fake_{b}"
    cand = []
    cand.append(os.path.join(d, pref_b) if d else pref_b)
    cand.append(f"fake_{orig_fname}")
    cand.append(pref_b)
    cand.append(orig_fname)
    seen=set(); out=[]
    for c in cand:
        if c not in seen:
            seen.add(c); out.append(c)
    return out


In [20]:
def build_diffs_annotation_relevance(confirmed_csv, annotations_root, real_csv, fake_csv, synonyms_map=None):
    """
    Returns:
      diffs_relevant   : np.array of (real - fake) over CONFIRMED neurons that are RELEVANT by annotation phrases
      diffs_irrelevant : np.array of (real - fake) over CONFIRMED neurons that are NOT relevant by annotation
      skipped          : count of pairs skipped due to missing/parse errors
    """
    confirmed = load_confirmed_neuron_labels(confirmed_csv, synonyms_map=synonyms_map)

    real_header, real_rows = load_csv_as_dicts(real_csv)
    fake_header, fake_rows = load_csv_as_dicts(fake_csv)

    real_fn = find_filename_column(real_header)
    fake_fn = find_filename_column(fake_header)
    
    for r in fake_rows:
        if r[fake_fn].startswith("fake_"):
            r[fake_fn] = r[fake_fn].replace("fake_", "", 1)

    
    neuron_cols = ensure_neuron_columns(real_header, 64)

    # index fake rows by filename
    fake_idx = { r[fake_fn]: r for r in fake_rows }

    # image list and object phrases
    real_fns = [r[real_fn] for r in real_rows]
    
    img2objs = load_image_objects_map(annotations_root, real_fns, synonyms_map=synonyms_map)

    diffs_relevant, diffs_irrelevant = [], []
    skipped = 0

    for rr in real_rows:
        fname = rr[real_fn]

        # match fake row via heuristic keys
        fr = None
        for key in _candidate_fake_keys(fname):
            fr = fake_idx.get(key)
            if fr is not None:
                break
        if fr is None:
            skipped += 1
            continue

        # activations
        try:
            rv = np.array([float(rr[c]) for c in neuron_cols], dtype=np.float64)
            fv = np.array([float(fr[c]) for c in neuron_cols], dtype=np.float64)
        except Exception:
            skipped += 1
            continue

        # WHOLE-PHRASE relevance by annotation
        objs = img2objs.get(fname, set())
        for j, labset in confirmed.items():
        
            # ZERO FILTERING: keep only positions where BOTH are nonzero
            if not (rv[j] > 0 and fv[j] > 0):
                continue
        
            is_rel = len(labset & objs) > 0
            diff = rv[j] - fv[j]
            if is_rel:
                diffs_relevant.append(diff)
            else:
                diffs_irrelevant.append(diff)


    return np.asarray(diffs_relevant, float), np.asarray(diffs_irrelevant, float), skipped


In [21]:
# Requires: pip install scipy
from scipy.stats import wilcoxon, rankdata

def wilcoxon_one_sided_summary(diffs, alternative):
    diffs = np.asarray(diffs, float)
    n_total = diffs.size
    nz = diffs != 0
    n_nonzero = int(nz.sum())
    n_zero = n_total - n_nonzero

    if n_nonzero == 0:
        return {
            "n_total": n_total, "n_nonzero": 0, "n_zero": n_zero,
            "stat": np.nan, "p": 1.0, "median_diff": 0.0, "mean_diff": 0.0,
            "prop_pos": 0.0, "r_rb": np.nan, "alternative": alternative
        }

    stat, p = wilcoxon(diffs, alternative=alternative, zero_method="wilcox", correction=False, mode="auto")

    d_nz = diffs[nz]
    ranks = rankdata(np.abs(d_nz), method="average")
    W_plus  = ranks[d_nz > 0].sum()
    W_minus = ranks[d_nz < 0].sum()
    denom = n_nonzero * (n_nonzero + 1) / 2.0
    r_rb = (W_plus - W_minus) / denom if denom > 0 else np.nan

    return {
        "n_total": n_total, "n_nonzero": n_nonzero, "n_zero": n_zero,
        "stat": float(stat), "p": float(p),
        "median_diff": float(np.median(diffs)), "mean_diff": float(np.mean(diffs)),
        "prop_pos": float(np.mean(diffs > 0)),
        "r_rb": float(r_rb), "alternative": alternative
    }

def print_summary(label, res, alpha=0.01):
    print(f"=== {label} ===")
    print(f"N total={res['n_total']}  nonzero={res['n_nonzero']}  zeros={res['n_zero']}")
    print(f"median(diff)={res['median_diff']:.4f}  mean(diff)={res['mean_diff']:.4f}  prop(diff>0)={res['prop_pos']:.3f}")
    print(f"Wilcoxon one-sided ({res['alternative']}): stat={res['stat']:.6g}, p={res['p']:.3e}")
    print(f"rank-biserial r_rb={res['r_rb']:.3f}  ( >0 favors real; <0 favors fake )")
    print("Decision:", "REJECT H0" if res["p"] < alpha else "fail to reject H0", f"at alpha={alpha}\n")


In [22]:
# Build diffs using WHOLE-PHRASE annotation-defined relevance
diffs_rel, diffs_irr, skipped = build_diffs_annotation_relevance(
    confirmed_csv=CONFIRMED_CSV,
    annotations_root=ANNOTATIONS_DIR,
    real_csv=REAL_ACT_CSV,
    fake_csv=FAKE_ACT_CSV,
    synonyms_map=SYNONYMS,
)
print(f"[info] built diffs: relevant={diffs_rel.size}, irrelevant={diffs_irr.size}, skipped_pairs={skipped}")

# H1: relevant ↓ in fakes  -> alternative='greater' on (real - fake)
res_H1 = wilcoxon_one_sided_summary(diffs_rel, alternative="greater")

# H2: irrelevant ↑ in fakes-> alternative='less'    on (real - fake)
res_H2 = wilcoxon_one_sided_summary(diffs_irr, alternative="less")

print_summary("H1 (relevant: real > fake)", res_H1, alpha=0.01)
print_summary("H2 (irrelevant: real < fake)", res_H2, alpha=0.01)


[info] built diffs: relevant=885, irrelevant=5638, skipped_pairs=0
=== H1 (relevant: real > fake) ===
N total=885  nonzero=885  zeros=0
median(diff)=1.0429  mean(diff)=1.2023  prop(diff>0)=0.740
Wilcoxon one-sided (greater): stat=326207, p=5.839e-66
rank-biserial r_rb=0.664  ( >0 favors real; <0 favors fake )
Decision: REJECT H0 at alpha=0.01

=== H2 (irrelevant: real < fake) ===
N total=5638  nonzero=5638  zeros=0
median(diff)=0.1645  mean(diff)=0.2790  prop(diff>0)=0.559
Wilcoxon one-sided (less): stat=9.4732e+06, p=1.000e+00
rank-biserial r_rb=0.192  ( >0 favors real; <0 favors fake )
Decision: fail to reject H0 at alpha=0.01



# --- Plan B: relevance = confirmed neurons that FIRE on the REAL image ---

In [30]:
# --- Plan B: relevance = confirmed neurons that FIRE on the REAL image ---


CONFIRMED_CSV   = "verification_summary_1 (test).csv"  # neuron_id, ecii concepts
# ANNOTATIONS_DIR = "test_annotations"                     # root folder with class subdirs of SUN2012 XMLs
REAL_ACT_CSV    = "preds_of_64Neurons_denseLayer_test.csv"                    # 64 cols "0".."63" + filenames
FAKE_ACT_CSV    = "[img+obj_labels_to_fakes]fake_activations(test).csv"       


THRESH_RATIO  = 0.8  # per-neuron threshold = ratio * max_real_activation
ALPHA         = 0.01


In [24]:
import os, csv
import numpy as np

def load_csv_as_dicts(path):
    with open(path, "r", newline="", encoding="utf-8") as f:
        rdr = csv.DictReader(f)
        rows = [r for r in rdr]
        header = rdr.fieldnames
    return header, rows

def find_filename_column(header):
    for cand in ("filenames","filename","file","path"):
        if cand in header: return cand
    raise RuntimeError("No filename column found (filenames/filename/file/path).")

def ensure_neuron_columns(header, n=64):
    cols = [str(i) for i in range(n)]
    missing = [c for c in cols if c not in header]
    if missing:
        raise RuntimeError(f"Missing neuron columns, e.g., {missing[:5]}")
    return cols

def _candidate_fake_keys(orig_fname):
    """
    Try to match real filename to fake filename variants:
      1) dir/fake_basename
      2) fake_<relpath>
      3) fake_basename
      4) original relpath
    """
    d = os.path.dirname(orig_fname)
    b = os.path.basename(orig_fname)
    pref_b = f"fake_{b}"
    cand = []
    cand.append(os.path.join(d, pref_b) if d else pref_b)
    cand.append(f"fake_{orig_fname}")
    cand.append(pref_b)
    cand.append(orig_fname)
    seen=set(); out=[]
    for c in cand:
        if c not in seen:
            seen.add(c); out.append(c)
    return out


In [25]:
def load_confirmed_neuron_ids(confirmed_csv_path,
                              id_col_candidates=("neuron_id","neuron","id")):
    ids = set()
    with open(confirmed_csv_path, "r", newline="", encoding="utf-8") as f:
        rdr = csv.DictReader(f)
        header = rdr.fieldnames or []
        id_col = next((c for c in id_col_candidates if c in header), None)
        if id_col is None:
            raise RuntimeError(f"Missing neuron ID column in {confirmed_csv_path}")
        for row in rdr:
            try:
                j = int(float(row[id_col]))
                if 0 <= j < 64:
                    ids.add(j)
            except:
                continue
    if not ids:
        raise RuntimeError("No confirmed neuron IDs found.")
    return sorted(ids)


In [26]:
def compute_threshold_from_real(real_rows, neuron_cols, ratio=0.8):
    first = True
    max_act = None
    for r in real_rows:
        vals = np.array([float(r[c]) for c in neuron_cols], dtype=np.float64)
        if first:
            max_act = vals; first = False
        else:
            max_act = np.maximum(max_act, vals)
    if max_act is None:
        raise RuntimeError("Real CSV had no rows.")
    return ratio * max_act  # shape (64,)


In [34]:
def build_diffs_real_firing_relevance(confirmed_csv, real_csv, fake_csv, ratio=0.8):
    """
    Relevant(i) = { confirmed j : real[i,j] >= T_j }, where T_j = ratio * max_real(j)
    Irrelevant(i) = confirmed \ Relevant(i)

    Returns:
      diffs_relevant   : np.array of (real - fake) for all (i,j) in Relevant(i)
      diffs_irrelevant : np.array of (real - fake) for all (i,j) in Irrelevant(i)
      skipped          : count of images skipped due to missing/parse issues
    """
    # confirmed IDs
    confirmed_ids = load_confirmed_neuron_ids(confirmed_csv)

    # load real/fake CSVs
    real_header, real_rows = load_csv_as_dicts(real_csv)
    fake_header, fake_rows = load_csv_as_dicts(fake_csv)

    real_fn_col = find_filename_column(real_header)
    fake_fn_col = find_filename_column(fake_header)

    for r in fake_rows:
        if r[fake_fn_col].startswith("fake_"):
            r[fake_fn_col] = r[fake_fn_col].replace("fake_", "", 1)
            
    neuron_cols = ensure_neuron_columns(real_header, 64)

    # thresholds from REAL
    T = compute_threshold_from_real(real_rows, neuron_cols, ratio=ratio)  # shape (64,)

    # index fake rows by filename
    fake_idx = { r[fake_fn_col]: r for r in fake_rows }

    diffs_relevant, diffs_irrelevant = [], []
    skipped = 0

    for rr in real_rows:
        fname = rr[real_fn_col]
        # find matching fake row
        fr = None
        for key in _candidate_fake_keys(fname):
            fr = fake_idx.get(key)
            if fr is not None: break
        if fr is None:
            skipped += 1
            continue

        try:
            rv = np.array([float(rr[c]) for c in neuron_cols], dtype=np.float64)
            fv = np.array([float(fr[c]) for c in neuron_cols], dtype=np.float64)
        except Exception:
            skipped += 1
            continue

        # split confirmed neurons by real-firing (rv[j] >= T[j])
        for j in confirmed_ids:

    # ZERO FILTERING: keep only if BOTH real and fake are nonzero
            if not (rv[j] > 0 and fv[j] > 0):
                continue
        
            diff = rv[j] - fv[j]
            if rv[j] >= T[j]:
                diffs_relevant.append(diff)    # relevant set (should decrease in fakes)
            else:
                diffs_irrelevant.append(diff)  # irrelevant set (hypothesis: increase in fakes)

    return np.asarray(diffs_relevant, float), np.asarray(diffs_irrelevant, float), skipped, T, confirmed_ids


In [35]:
# pip install scipy
from scipy.stats import wilcoxon, rankdata

def wilcoxon_one_sided_summary(diffs, alternative):
    diffs = np.asarray(diffs, float)
    n_total = diffs.size
    nz = diffs != 0
    n_nonzero = int(nz.sum())
    n_zero = n_total - n_nonzero

    if n_nonzero == 0:
        return {
            "n_total": n_total, "n_nonzero": 0, "n_zero": n_zero,
            "stat": np.nan, "p": 1.0, "median_diff": 0.0, "mean_diff": 0.0,
            "prop_pos": 0.0, "r_rb": np.nan, "alternative": alternative
        }

    stat, p = wilcoxon(diffs, alternative=alternative, zero_method="wilcox", correction=False, mode="auto")

    d_nz = diffs[nz]
    ranks = rankdata(np.abs(d_nz), method="average")
    W_plus  = ranks[d_nz > 0].sum()
    W_minus = ranks[d_nz < 0].sum()
    denom = n_nonzero * (n_nonzero + 1) / 2.0
    r_rb = (W_plus - W_minus) / denom if denom > 0 else np.nan

    return {
        "n_total": n_total, "n_nonzero": n_nonzero, "n_zero": n_zero,
        "stat": float(stat), "p": float(p),
        "median_diff": float(np.median(diffs)), "mean_diff": float(np.mean(diffs)),
        "prop_pos": float(np.mean(diffs > 0)),
        "r_rb": float(r_rb), "alternative": alternative
    }

def print_summary(label, res, alpha=0.01):
    print(f"=== {label} ===")
    print(f"N total={res['n_total']}  nonzero={res['n_nonzero']}  zeros={res['n_zero']}")
    print(f"median(diff)={res['median_diff']:.4f}  mean(diff)={res['mean_diff']:.4f}  prop(diff>0)={res['prop_pos']:.3f}")
    print(f"Wilcoxon one-sided ({res['alternative']}): stat={res['stat']:.6g}, p={res['p']:.3e}")
    print(f"rank-biserial r_rb={res['r_rb']:.3f}  ( >0 favors real; <0 favors fake )")
    print("Decision:", "REJECT H0" if res["p"] < alpha else "fail to reject H0", f"at alpha={alpha}\n")


In [36]:
diffs_rel, diffs_irr, skipped, T, confirmed_ids = build_diffs_real_firing_relevance(
    confirmed_csv=CONFIRMED_CSV,
    real_csv=REAL_ACT_CSV,
    fake_csv=FAKE_ACT_CSV,
    ratio=THRESH_RATIO,
)
print(f"[info][Plan B] diffs built: relevant={diffs_rel.size}, irrelevant={diffs_irr.size}, skipped_pairs={skipped}")

# H1: relevant ↓ in fakes  -> alternative='greater' on (real - fake)
res_H1 = wilcoxon_one_sided_summary(diffs_rel, alternative="greater")

# H2: irrelevant ↑ in fakes-> alternative='less'    on (real - fake)
res_H2 = wilcoxon_one_sided_summary(diffs_irr, alternative="less")

print_summary("H1 (relevant by real-firing: real > fake)", res_H1, alpha=ALPHA)
print_summary("H2 (irrelevant by real-firing: real < fake)", res_H2, alpha=ALPHA)


[info][Plan B] diffs built: relevant=249, irrelevant=6274, skipped_pairs=0
=== H1 (relevant by real-firing: real > fake) ===
N total=249  nonzero=249  zeros=0
median(diff)=2.2209  mean(diff)=2.5786  prop(diff>0)=0.924
Wilcoxon one-sided (greater): stat=30642, p=2.117e-40
rank-biserial r_rb=0.969  ( >0 favors real; <0 favors fake )
Decision: REJECT H0 at alpha=0.01

=== H2 (irrelevant by real-firing: real < fake) ===
N total=6274  nonzero=6274  zeros=0
median(diff)=0.1967  mean(diff)=0.3180  prop(diff>0)=0.570
Wilcoxon one-sided (less): stat=1.20596e+07, p=1.000e+00
rank-biserial r_rb=0.225  ( >0 favors real; <0 favors fake )
Decision: fail to reject H0 at alpha=0.01



# --- Per-image aggregation for Plan A 

In [51]:
# --- Per-image aggregation for Plan A (annotation-based relevance, WHOLE-PHRASE) ---

import numpy as np

def build_imagewise_aggregates_annotation_relevance(confirmed_csv,
                                                    annotations_root,
                                                    real_csv,
                                                    fake_csv,
                                                    agg="mean",
                                                    eps=None,
                                                    synonyms_map=None):

    def _aggregate(arr, how):
        if arr.size == 0: return np.nan
        return float(np.mean(arr)) if how == "mean" else float(np.median(arr))

    # 1) confirmed labels as WHOLE PHRASES per neuron
    confirmed = load_confirmed_neuron_labels(confirmed_csv, synonyms_map=synonyms_map)

    # 2) activations CSVs
    real_header, real_rows = load_csv_as_dicts(real_csv)
    fake_header, fake_rows = load_csv_as_dicts(fake_csv)
    real_fn = find_filename_column(real_header)
    fake_fn = find_filename_column(fake_header)

    # for r in fake_rows:
    #     if r[fake_fn].startswith("fake_"):
    #         r[fake_fn] = r[fake_fn].replace("fake_", "", 1)
    
    neuron_cols = ensure_neuron_columns(real_header, 64)

    # 3) index fake by filename
    fake_idx = { r[fake_fn]: r for r in fake_rows }

    # 4) per-image object phrases from XMLs
    real_fns = [r[real_fn] for r in real_rows]
    img2objs = load_image_objects_map(annotations_root, real_fns, synonyms_map=synonyms_map)

    image_ids, agg_rel, agg_irr = [], [], []

    # 5) iterate images
    for rr in real_rows:
        fname = rr[real_fn]
        # match fake
        fr = None
        for key in _candidate_fake_keys(fname):
            fr = fake_idx.get(key)
            if fr is not None:
                break
        if fr is None:
            continue

        # read vectors
        try:
            rv = np.array([float(rr[c]) for c in neuron_cols], dtype=np.float64)
            fv = np.array([float(fr[c]) for c in neuron_cols], dtype=np.float64)
        except Exception:
            continue

        objs = img2objs.get(fname, set())

        diffs_rel, diffs_irr = [], []
        for j, labset in confirmed.items():
            # WHOLE-PHRASE relevance: intersection non-empty
            is_rel = len(labset & objs) > 0
            # optional small-activation filter
            if eps is not None and max(rv[j], fv[j]) < eps:
                continue
            d = rv[j] - fv[j]
            (diffs_rel if is_rel else diffs_irr).append(d)

        diffs_rel = np.asarray(diffs_rel, float)
        diffs_irr = np.asarray(diffs_irr, float)

        image_ids.append(fname)
        agg_rel.append(_aggregate(diffs_rel, agg))
        agg_irr.append(_aggregate(diffs_irr, agg))

    return image_ids, np.asarray(agg_rel, float), np.asarray(agg_irr, float)


In [53]:
# Config (phrase-level Plan A)
ALPHA = 0.05
AGG   = "mean"      # or "median"
EPS   = None        # e.g., 0.05 to drop tiny-activation pairs
# SYNONYMS already defined as phrase-level map and normalized

img_ids_A, img_agg_rel_A, img_agg_irr_A = build_imagewise_aggregates_annotation_relevance(
    confirmed_csv=CONFIRMED_CSV,
    annotations_root=ANNOTATIONS_DIR,
    real_csv=REAL_ACT_CSV,
    fake_csv=FAKE_ACT_CSV,
    agg=AGG,
    eps=EPS,
    synonyms_map=SYNONYMS,
)
print(f"[Plan A] images={len(img_ids_A)}  with_agg_rel(non-NaN)={np.sum(~np.isnan(img_agg_rel_A))}  with_agg_irr(non-NaN)={np.sum(~np.isnan(img_agg_irr_A))}")

# H1: relevant ↓ in fakes -> alternative='greater' on image-level aggregates
res_H1_img_A = wilcoxon_imagelevel(img_agg_rel_A, alternative="greater")

# H2: irrelevant ↑ in fakes -> alternative='less' on image-level aggregates
res_H2_img_A = wilcoxon_imagelevel(img_agg_irr_A, alternative="less")

print_img_summary("Plan A — H1 (per-image, relevant: real > fake)", res_H1_img_A, alpha=ALPHA)
print_img_summary("Plan A — H2 (per-image, irrelevant: real < fake)", res_H2_img_A, alpha=ALPHA)


[Plan A] images=3157  with_agg_rel(non-NaN)=1908  with_agg_irr(non-NaN)=3157
=== Plan A — H1 (per-image, relevant: real > fake) ===
Images used       : total=1908  nonzero=1853  zeros=55
median=0.3357  mean=0.3407  prop(>0)=0.562
Wilcoxon one-sided: stat=1.04318e+06, p=6.153e-16
rank-biserial r_rb: 0.215  (>0 favors real)
Decision: REJECT H0 at alpha=0.05

=== Plan A — H2 (per-image, irrelevant: real < fake) ===
Images used       : total=3157  nonzero=3157  zeros=0
median=0.0887  mean=0.0880  prop(>0)=0.626
Wilcoxon one-sided: stat=3.31442e+06, p=1.000e+00
rank-biserial r_rb: 0.330  (>0 favors real)
Decision: fail to reject H0 at alpha=0.05



# --- Per-image aggregation for Plan B (real-firing relevance) ---

In [48]:
# --- Per-image aggregation for Plan B (real-firing relevance) ---

import os, csv
import numpy as np
from scipy.stats import wilcoxon, rankdata

# Reuse your existing helpers (_candidate_fake_keys, load_csv_as_dicts, find_filename_column, ensure_neuron_columns)
# and load_confirmed_neuron_ids, compute_threshold_from_real

def build_imagewise_aggregates_real_firing(confirmed_csv, real_csv, fake_csv,
                                           ratio=0.8, agg="mean", eps=None):
    """
    For each image i:
      Relevant(i)   = { confirmed j : real[i,j] >= T_j }
      Irrelevant(i) = confirmed \ Relevant(i)
    Compute per-image aggregates over diffs d_ij = real[i,j] - fake[i,j]:
      agg_rel[i] = mean/median of d_ij over j in Relevant(i)   (skip if empty)
      agg_irr[i] = mean/median of d_ij over j in Irrelevant(i) (skip if empty)
    Optionally filter pairs by activation magnitude: keep only if max(real,fake) >= eps
    Returns:
      image_ids: list of filenames used
      agg_rel:   np.array of per-image aggregates for relevant (same order as image_ids, NaN if none)
      agg_irr:   np.array of per-image aggregates for irrelevant (NaN if none)
      T:         thresholds vector (64,)
      confirmed_ids: list of confirmed neuron IDs
    """
    def _aggregate(arr, how):
        if arr.size == 0: return np.nan
        return np.mean(arr) if how == "mean" else np.median(arr)

    # Load confirmed
    confirmed_ids = load_confirmed_neuron_ids(confirmed_csv)

    # Load CSVs
    real_header, real_rows = load_csv_as_dicts(real_csv)
    fake_header, fake_rows = load_csv_as_dicts(fake_csv)
    real_fn = find_filename_column(real_header)
    fake_fn = find_filename_column(fake_header)
    neuron_cols = ensure_neuron_columns(real_header, 64)

    # Thresholds T_j from REAL
    T = compute_threshold_from_real(real_rows, neuron_cols, ratio=ratio)

    # Index fake by filename
    fake_idx = { r[fake_fn]: r for r in fake_rows }

    image_ids, agg_rel, agg_irr = [], [], []

    for rr in real_rows:
        fname = rr[real_fn]

        # Match fake
        fr = None
        for key in _candidate_fake_keys(fname):
            fr = fake_idx.get(key)
            if fr is not None:
                break
        if fr is None:
            continue

        # Vectors
        try:
            rv = np.array([float(rr[c]) for c in neuron_cols], dtype=np.float64)
            fv = np.array([float(fr[c]) for c in neuron_cols], dtype=np.float64)
        except Exception:
            continue

        diffs_rel = []
        diffs_irr = []

        for j in confirmed_ids:
            # Optional small-activation filter
            if eps is not None and max(rv[j], fv[j]) < eps:
                continue

            d = rv[j] - fv[j]
            if rv[j] >= T[j]:
                diffs_rel.append(d)
            else:
                diffs_irr.append(d)

        diffs_rel = np.asarray(diffs_rel, float)
        diffs_irr = np.asarray(diffs_irr, float)

        image_ids.append(fname)
        agg_rel.append(_aggregate(diffs_rel, agg))
        agg_irr.append(_aggregate(diffs_irr, agg))

    return image_ids, np.asarray(agg_rel, float), np.asarray(agg_irr, float), T, confirmed_ids


In [49]:
def wilcoxon_imagelevel(arr, alternative):
    """
    Runs one-sided Wilcoxon on image-level aggregates (drops NaNs and zeros).
    Tests median(arr) > 0 if alternative='greater', or < 0 if 'less'.
    """
    arr = np.asarray(arr, float)
    arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return {"n_total": 0, "n_nonzero": 0, "n_zero": 0, "stat": np.nan, "p": 1.0,
                "median": np.nan, "mean": np.nan, "prop_pos": np.nan, "r_rb": np.nan}
    nz = arr != 0
    n_total = arr.size
    n_nonzero = int(nz.sum())
    n_zero = n_total - n_nonzero

    if n_nonzero == 0:
        return {"n_total": n_total, "n_nonzero": 0, "n_zero": n_zero, "stat": np.nan, "p": 1.0,
                "median": float(np.median(arr)), "mean": float(np.mean(arr)),
                "prop_pos": float(np.mean(arr > 0)), "r_rb": np.nan}

    stat, p = wilcoxon(arr, alternative=alternative, zero_method="wilcox", correction=False, mode="auto")

    # rank-biserial on nonzero
    d_nz = arr[nz]
    ranks = rankdata(np.abs(d_nz), method="average")
    W_plus = ranks[d_nz > 0].sum()
    W_minus = ranks[d_nz < 0].sum()
    denom = n_nonzero * (n_nonzero + 1) / 2.0
    r_rb = (W_plus - W_minus) / denom if denom > 0 else np.nan

    return {
        "n_total": n_total, "n_nonzero": n_nonzero, "n_zero": n_zero,
        "stat": float(stat), "p": float(p),
        "median": float(np.median(arr)), "mean": float(np.mean(arr)),
        "prop_pos": float(np.mean(arr > 0)), "r_rb": float(r_rb)
    }

def print_img_summary(title, res, alpha=0.01):
    print(f"=== {title} ===")
    print(f"Images used       : total={res['n_total']}  nonzero={res['n_nonzero']}  zeros={res['n_zero']}")
    print(f"median={res['median']:.4f}  mean={res['mean']:.4f}  prop(>0)={res['prop_pos']:.3f}")
    print(f"Wilcoxon one-sided: stat={res['stat']:.6g}, p={res['p']:.3e}")
    print(f"rank-biserial r_rb: {res['r_rb']:.3f}  (>0 favors real)")
    print("Decision:", "REJECT H0" if res["p"] < alpha else "fail to reject H0", f"at alpha={alpha}\n")


In [50]:
# Config
THRESH_RATIO = 0.8     # relevance threshold (real-firing)
ALPHA = 0.01
AGG = "mean"           # or "median"
EPS = None             # e.g., 0.05 to ignore tiny-activation pairs

# Build per-image aggregates
img_ids, img_agg_rel, img_agg_irr, T, confirmed_ids = build_imagewise_aggregates_real_firing(
    confirmed_csv=CONFIRMED_CSV,
    real_csv=REAL_ACT_CSV,
    fake_csv=FAKE_ACT_CSV,
    ratio=THRESH_RATIO,
    agg=AGG,
    eps=EPS,
)
print(f"[info] images={len(img_ids)}  with_agg_rel(non-NaN)={np.sum(~np.isnan(img_agg_rel))}  with_agg_irr(non-NaN)={np.sum(~np.isnan(img_agg_irr))}")

# H1: relevant ↓ in fakes -> alternative='greater' on per-image aggregates
res_H1_img = wilcoxon_imagelevel(img_agg_rel, alternative="greater")

# H2: irrelevant ↑ in fakes -> alternative='less' on per-image aggregates
res_H2_img = wilcoxon_imagelevel(img_agg_irr, alternative="less")

print_img_summary("H1 (per-image, relevant: real > fake)", res_H1_img, alpha=ALPHA)
print_img_summary("H2 (per-image, irrelevant: real < fake)", res_H2_img, alpha=ALPHA)


[info] images=3157  with_agg_rel(non-NaN)=499  with_agg_irr(non-NaN)=3157
=== H1 (per-image, relevant: real > fake) ===
Images used       : total=499  nonzero=499  zeros=0
median=3.1930  mean=3.2310  prop(>0)=0.974
Wilcoxon one-sided: stat=124378, p=8.595e-83
rank-biserial r_rb: 0.994  (>0 favors real)
Decision: REJECT H0 at alpha=0.01

=== H2 (per-image, irrelevant: real < fake) ===
Images used       : total=3157  nonzero=3157  zeros=0
median=0.1003  mean=0.0907  prop(>0)=0.621
Wilcoxon one-sided: stat=3.32065e+06, p=1.000e+00
rank-biserial r_rb: 0.332  (>0 favors real)
Decision: fail to reject H0 at alpha=0.01



## test

In [1]:
import os, csv, numpy as np

# --- edit paths ---
CONFIRMED_CSV = "verification_combine_d(summary_1).csv"  # has neuron_id
REAL_CSV      = "preds_of_64Neurons_denseLayer_training.csv"                   # cols "0".."63" + filenames
FAKE_CSV      = "[obj_labels_to_fakes]fake_activations.csv"                   # same schema; fake_<...>.jpg names
THRESH_RATIO  = 0.8
EPSILON       = 0.0
OUT_CSV       = "irrelevant_changes_planB.csv"  # or None to skip writing

# --- tiny helpers ---
def read_csv(path):
    with open(path, "r", newline="", encoding="utf-8") as f:
        r = csv.DictReader(f); rows = list(r); hdr = r.fieldnames
    return hdr, rows

def fname_col(hdr):
    for c in ("filenames","filename","file","path"):
        if c in hdr: return c
    raise RuntimeError("No filename column.")

def confirmed_ids(path):
    hdr, rows = read_csv(path)
    for c in ("neuron_id","neuron","id"):
        if c in hdr: idc = c; break
    ids = []
    for row in rows:
        try:
            j = int(float(row[idc]))
            if 0 <= j < 64: ids.append(j)
        except: pass
    if not ids: raise RuntimeError("No confirmed IDs.")
    return sorted(set(ids))

def cand_fake_keys(fn):
    d, b = os.path.dirname(fn), os.path.basename(fn)
    k = [os.path.join(d, f"fake_{b}") if d else f"fake_{b}",
         f"fake_{fn}", f"fake_{b}", fn]
    out, seen = [], set()
    for x in k:
        if x not in seen: seen.add(x); out.append(x)
    return out

# --- load everything ---
cids = confirmed_ids(CONFIRMED_CSV)
hdrR, rowsR = read_csv(REAL_CSV)
hdrF, rowsF = read_csv(FAKE_CSV)
fnR, fnF = fname_col(hdrR), fname_col(hdrF)
cols = [str(i) for i in range(64)]
fake_idx = {r[fnF]: r for r in rowsF}

# thresholds T_j = 0.8 * max_real(j)
T = None
for r in rowsR:
    v = np.array([float(r[c]) for c in cols], float)
    T = v if T is None else np.maximum(T, v)
T = THRESH_RATIO * T

# --- enumerate irrelevant cases & compare ---
inc, opp, out_rows, skipped = 0, 0, [], 0
for rr in rowsR:
    fn = rr[fnR]
    fr = None
    for k in cand_fake_keys(fn):
        fr = fake_idx.get(k)
        if fr is not None: break
    if fr is None: skipped += 1; continue

    rv = np.array([float(rr[c]) for c in cols], float)
    fv = np.array([float(fr[c]) for c in cols], float)

    for j in cids:
        if rv[j] >= T[j]: 
            continue  # relevant -> skip; we want irrelevant only
        r, f, d = float(rv[j]), float(fv[j]), float(rv[j]-fv[j])
        if f - r > EPSILON:
            inc += 1
            if OUT_CSV: out_rows.append({"filenames": fn, "neuron_id": j, "real": r, "fake": f, "diff_real_minus_fake": d, "flag":"fake>real"})
        elif r - f > EPSILON:
            opp += 1
            if OUT_CSV: out_rows.append({"filenames": fn, "neuron_id": j, "real": r, "fake": f, "diff_real_minus_fake": d, "flag":"real>fake"})

print(f"[PlanB][irrelevant confirmed] fake>real: {inc}  |  real>fake: {opp}  |  skipped pairs: {skipped}")

if OUT_CSV:
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["filenames","neuron_id","real","fake","diff_real_minus_fake","flag"])
        w.writeheader(); w.writerows(out_rows)
    print(f"[PlanB] wrote details -> {OUT_CSV}")


[PlanB][irrelevant confirmed] fake>real: 23737  |  real>fake: 27264  |  skipped pairs: 0
[PlanB] wrote details -> irrelevant_changes_planB.csv


## check zero value in irrelevant confirmed neurons for fakes

In [72]:
import csv
import numpy as np

def report_irrelevant_fake_zero(
    confirmed_csv,
    real_csv,
    fake_csv,
    ratio=0.8,
    zero_eps=0.0,   # treat fv <= zero_eps as "fake == 0"
    real_eps=0.0,   # treat rv > real_eps as "real != 0"
    out_csv_path=None
):
    """
    Plan B: Irrelevant = confirmed neuron that does NOT fire on the REAL image (rv[j] < T[j])

    Reports:
      - All pairs where irrelevant AND (fake <= zero_eps)
      - Of those, how many also satisfy (real > real_eps)  <-- your stricter condition

    Prints totals and basic stats of the real activations for those pairs.
    Optionally writes a CSV with columns: filenames, neuron_id, real_act, fake_act, real_nonzero_flag
    """
    # Load confirmed IDs
    confirmed_ids = load_confirmed_neuron_ids(confirmed_csv)

    # Load CSVs
    real_header, real_rows = load_csv_as_dicts(real_csv)
    fake_header, fake_rows = load_csv_as_dicts(fake_csv)
    real_fn = find_filename_column(real_header)
    fake_fn = find_filename_column(fake_header)
    neuron_cols = ensure_neuron_columns(real_header, 64)

    # Per-neuron thresholds from REAL
    T = compute_threshold_from_real(real_rows, neuron_cols, ratio=ratio)

    # Index fakes
    fake_idx = { r[fake_fn]: r for r in fake_rows }

    # Collections
    rows_all = []     # all irrelevant & fake==0
    rows_strict = []  # subset with real>real_eps & fake==0
    per_image_zero_counts = {}
    per_image_strict_counts = {}

    for rr in real_rows:
        fname = rr[real_fn]
        # match fake
        fr = None
        for key in _candidate_fake_keys(fname):
            fr = fake_idx.get(key)
            if fr is not None:
                break
        if fr is None:
            continue

        # vectors
        try:
            rv = np.array([float(rr[c]) for c in neuron_cols], dtype=np.float64)
            fv = np.array([float(fr[c]) for c in neuron_cols], dtype=np.float64)
        except Exception:
            continue

        c_all = 0
        c_strict = 0
        for j in confirmed_ids:
            # irrelevant by Plan B
            if rv[j] < T[j]:
                if fv[j] <= zero_eps:
                    c_all += 1
                    real_nz = (rv[j] > real_eps)
                    rows_all.append({
                        "filenames": fname,
                        "neuron_id": j,
                        "real_act": float(rv[j]),
                        "fake_act": float(fv[j]),
                        "real_nonzero": int(real_nz)
                    })
                    if real_nz:
                        c_strict += 1
                        rows_strict.append({
                            "filenames": fname,
                            "neuron_id": j,
                            "real_act": float(rv[j]),
                            "fake_act": float(fv[j])
                        })
        per_image_zero_counts[fname] = c_all
        per_image_strict_counts[fname] = c_strict

    # Summaries
    image_total = len(per_image_zero_counts)
    imgs_any_zero   = sum(1 for k,v in per_image_zero_counts.items() if v > 0)
    imgs_any_strict = sum(1 for k,v in per_image_strict_counts.items() if v > 0)

    real_vals_all    = np.array([r["real_act"] for r in rows_all],    dtype=np.float64) if rows_all    else np.array([], dtype=np.float64)
    real_vals_strict = np.array([r["real_act"] for r in rows_strict], dtype=np.float64) if rows_strict else np.array([], dtype=np.float64)

    print("=== Irrelevant confirmed neurons with fake==0 (Plan B) ===")
    print(f"Images processed                         : {image_total}")
    print(f"Images with ≥1 (fake==0)                 : {imgs_any_zero}")
    print(f"Images with ≥1 (real>real_eps & fake==0) : {imgs_any_strict}")
    print(f"Pairs (fake==0)                          : {len(rows_all)}")
    print(f"Pairs (real>real_eps & fake==0)          : {len(rows_strict)}")
    if real_vals_all.size:
        print(f"Real activations for (fake==0): mean={real_vals_all.mean():.4f}, median={np.median(real_vals_all):.4f}")
    if real_vals_strict.size:
        print(f"Real activations for (real>eps & fake==0): mean={real_vals_strict.mean():.4f}, median={np.median(real_vals_strict):.4f}")

    # Optional CSV dump (includes flag so you can filter later)
    if out_csv_path:
        with open(out_csv_path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["filenames","neuron_id","real_act","fake_act","real_nonzero"])
            w.writeheader()
            w.writerows(rows_all)
        print(f"[saved] detailed rows -> {out_csv_path}")

    return {
        "per_image_any_zero": per_image_zero_counts,
        "per_image_any_strict": per_image_strict_counts,
        "rows_all": rows_all,
        "rows_strict": rows_strict
    }


In [75]:
res = report_irrelevant_fake_zero(
    confirmed_csv="verification_combine_d(summary_1).csv",
    real_csv=REAL_ACT_CSV,
    fake_csv=FAKE_ACT_CSV,
    ratio=0.8,
    zero_eps=0.0,
    real_eps=0.0,
    out_csv_path="irrelevant_fake_zero_detail.csv",
)


=== Irrelevant confirmed neurons with fake==0 (Plan B) ===
Images processed                         : 3157
Images with ≥1 (fake==0)                 : 3157
Images with ≥1 (real>real_eps & fake==0) : 2930
Pairs (fake==0)                          : 59011
Pairs (real>real_eps & fake==0)          : 9677
Real activations for (fake==0): mean=0.1924, median=0.0000
Real activations for (real>eps & fake==0): mean=1.1730, median=0.8035
[saved] detailed rows -> irrelevant_fake_zero_detail.csv


In [76]:
import csv
import numpy as np

def report_irrelevant_fake_zero_planA(
    confirmed_csv,
    annotations_root,
    real_csv,
    fake_csv,
    zero_eps=0.0,   # treat fv <= zero_eps as "fake == 0"
    real_eps=0.0,   # treat rv > real_eps as "real != 0"
    out_csv_path=None,
    synonyms_map=None
):
    """
    Plan A (annotation-based relevance, WHOLE-PHRASE):
      Relevant_A(i)   = { confirmed j : label_phrases(j) ∩ objects_in_image(i) ≠ ∅ }
      Irrelevant_A(i) = confirmed \ Relevant_A(i)

    We find, for each image i and confirmed neuron j that is Irrelevant_A(i):
      - cases where fake activation fv[i,j] <= zero_eps
      - of those, the STRICT subset where real activation rv[i,j] > real_eps

    Prints summary and optionally writes a CSV:
      columns: filenames, neuron_id, real_act, fake_act, real_nonzero (0/1)

    Returns dict with:
      {
        "per_image_any_zero":   { filename: count_fake_zero_on_irrelevant },
        "per_image_any_strict": { filename: count_real>eps_and_fake_zero_on_irrelevant },
        "rows_all":   [ ... all hits ... ],
        "rows_strict":[ ... strict hits ... ]
      }
    """
    # 1) confirmed neuron labels as WHOLE PHRASES
    confirmed = load_confirmed_neuron_labels(confirmed_csv, synonyms_map=synonyms_map)
    confirmed_ids = sorted(confirmed.keys())

    # 2) load real/fake activations
    real_header, real_rows = load_csv_as_dicts(real_csv)
    fake_header, fake_rows = load_csv_as_dicts(fake_csv)
    real_fn = find_filename_column(real_header)
    fake_fn = find_filename_column(fake_header)
    neuron_cols = ensure_neuron_columns(real_header, 64)

    # 3) index fake by filename
    fake_idx = { r[fake_fn]: r for r in fake_rows }

    # 4) image -> set(WHOLE-PHRASE objects) from XML
    real_fns = [r[real_fn] for r in real_rows]
    img2objs = load_image_objects_map(annotations_root, real_fns, synonyms_map=synonyms_map)

    # 5) iterate images
    rows_all = []     # all irrelevant & fake==0
    rows_strict = []  # subset with real>real_eps & fake==0
    per_image_zero_counts = {}
    per_image_strict_counts = {}

    for rr in real_rows:
        fname = rr[real_fn]
        # match fake row
        fr = None
        for key in _candidate_fake_keys(fname):
            fr = fake_idx.get(key)
            if fr is not None:
                break
        if fr is None:
            continue

        # activations vector
        try:
            rv = np.array([float(rr[c]) for c in neuron_cols], dtype=np.float64)
            fv = np.array([float(fr[c]) for c in neuron_cols], dtype=np.float64)
        except Exception:
            continue

        objs = img2objs.get(fname, set())

        c_all = 0
        c_strict = 0
        for j in confirmed_ids:
            labset = confirmed[j]              # set of WHOLE-PHRASE labels for neuron j
            is_relevant = len(labset & objs) > 0
            if is_relevant:
                continue  # we only inspect IRRELEVANT confirmed neurons

            if fv[j] <= zero_eps:
                c_all += 1
                real_nz = (rv[j] > real_eps)
                rows_all.append({
                    "filenames": fname,
                    "neuron_id": j,
                    "real_act": float(rv[j]),
                    "fake_act": float(fv[j]),
                    "real_nonzero": int(real_nz)
                })
                if real_nz:
                    c_strict += 1
                    rows_strict.append({
                        "filenames": fname,
                        "neuron_id": j,
                        "real_act": float(rv[j]),
                        "fake_act": float(fv[j])
                    })

        per_image_zero_counts[fname] = c_all
        per_image_strict_counts[fname] = c_strict

    # 6) summaries
    image_total = len(per_image_zero_counts)
    imgs_any_zero   = sum(1 for _,v in per_image_zero_counts.items() if v > 0)
    imgs_any_strict = sum(1 for _,v in per_image_strict_counts.items() if v > 0)

    real_vals_all    = np.array([r["real_act"] for r in rows_all],    dtype=np.float64) if rows_all    else np.array([], dtype=np.float64)
    real_vals_strict = np.array([r["real_act"] for r in rows_strict], dtype=np.float64) if rows_strict else np.array([], dtype=np.float64)

    print("=== Plan A: Irrelevant confirmed neurons with fake==0 (phrase-match) ===")
    print(f"Images processed                         : {image_total}")
    print(f"Images with ≥1 (fake==0)                 : {imgs_any_zero}")
    print(f"Images with ≥1 (real>real_eps & fake==0) : {imgs_any_strict}")
    print(f"Pairs (fake==0)                          : {len(rows_all)}")
    print(f"Pairs (real>real_eps & fake==0)          : {len(rows_strict)}")
    if real_vals_all.size:
        print(f"Real activations for (fake==0): mean={real_vals_all.mean():.4f}, median={np.median(real_vals_all):.4f}")
    if real_vals_strict.size:
        print(f"Real activations for (real>eps & fake==0): mean={real_vals_strict.mean():.4f}, median={np.median(real_vals_strict):.4f}")

    # 7) optional CSV
    if out_csv_path:
        with open(out_csv_path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["filenames","neuron_id","real_act","fake_act","real_nonzero"])
            w.writeheader()
            w.writerows(rows_all)
        print(f"[saved] detailed rows -> {out_csv_path}")

    return {
        "per_image_any_zero": per_image_zero_counts,
        "per_image_any_strict": per_image_strict_counts,
        "rows_all": rows_all,
        "rows_strict": rows_strict
    }


In [77]:
resA = report_irrelevant_fake_zero_planA(
    confirmed_csv="verification_combine_d(summary_1).csv",
    annotations_root="sun_annotations_xml",
    real_csv=REAL_ACT_CSV,
    fake_csv=FAKE_ACT_CSV,
    zero_eps=0.0,      # exact zeros on fake; use e.g. 1e-8 if you want “near-zero”
    real_eps=0.0,      # strictly > 0 means “real nonzero”; use 1e-6 for numerical noise guard
    out_csv_path="planA_irrelevant_fake_zero_detail.csv",
    synonyms_map=SYNONYMS  # your whole-phrase synonym map (optional)
)

# per_img_any_zero_A   = resA["per_image_any_zero"]
# per_img_any_strict_A = resA["per_image_any_strict"]
# rows_all_A           = resA["rows_all"]
# rows_strict_A        = resA["rows_strict"]


=== Plan A: Irrelevant confirmed neurons with fake==0 (phrase-match) ===
Images processed                         : 3157
Images with ≥1 (fake==0)                 : 3157
Images with ≥1 (real>real_eps & fake==0) : 2931
Pairs (fake==0)                          : 59023
Pairs (real>real_eps & fake==0)          : 9689
Real activations for (fake==0): mean=0.1937, median=0.0000
Real activations for (real>eps & fake==0): mean=1.1799, median=0.8049
[saved] detailed rows -> planA_irrelevant_fake_zero_detail.csv
